# Visual Noise Ablation Study

**Research question:** At what point does visual degradation cause VLM performance to break down?

We apply a **10-level noise gradient** to rendered math problems:

| Level | Name | Description |
|-------|------|-------------|
| 0 | Clean | Original PIL rendering (baseline) |
| 1 | JPEG mild | Compression quality=50 |
| 2 | Blur light | Gaussian blur radius=1 |
| 3 | JPEG heavy | Compression quality=15 |
| 4 | Blur+noise | Blur r=2 + Gaussian noise std=15 |
| 5 | Heavy degradation | Blur r=3 + noise std=25 + contrast loss |
| 6 | Geometric | Rotation 3° + perspective skew + shadow |
| 7 | Handwriting | Handwriting-style font + paper texture |
| 8 | Screenshot | Browser chrome + anti-aliasing |
| 9 | Worst case | All combined: blur + noise + rotation + JPEG |

This reveals whether the accuracy drop is due to OCR failure or something deeper.

**Runtime:** ~30 min per noise level per model (using subset of 200 problems)

In [ ]:
!pip install torch transformers datasets scipy pyyaml tqdm pillow bitsandbytes --quiet

In [ ]:
!git clone https://github.com/Ro-netizen004/vlm-modality-research.git /content/repo 2>/dev/null || echo 'exists'
import sys
sys.path.insert(0, '/content/repo')

from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/vlm_research_results/noise_ablation'
!mkdir -p {OUTPUT_DIR}

In [ ]:
# ══════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════

NUM_PROBLEMS = 200       # subset for tractability (200 × 10 levels = 2000 inferences)
NOISE_LEVELS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  # all 10 levels

MODEL = {
    'name': 'Qwen/Qwen2-VL-2B-Instruct',
    'type': 'qwen',
    'max_new_tokens': 256,
    'torch_dtype': 'bfloat16',
    'quantize': None,
}

print(f'Model: {MODEL["name"]}')
print(f'Problems: {NUM_PROBLEMS}')
print(f'Noise levels: {NOISE_LEVELS}')
print(f'Total inferences: {NUM_PROBLEMS * len(NOISE_LEVELS)}')

In [ ]:
import os, time, json, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm import tqdm

from src.models import VLMModel
from src.evaluation import answers_match, classify_error, compute_accuracy, bootstrap_ci, binomial_ci
from src.rendering import render_text_to_image
from src.noise import (
    apply_noise_level, render_noisy_images,
    NOISE_LEVELS as NOISE_CONFIGS,
    get_noise_level_names, get_noise_level_descriptions,
)

torch.manual_seed(42)

# Load dataset
ds = load_dataset('gsm8k', 'main', split='test').select(range(NUM_PROBLEMS))
questions = list(ds['question'])
references = list(ds['answer'])
N = len(questions)
print(f'Loaded {N} problems')

In [ ]:
# ── Render all noise levels ──
IMAGE_DIR = os.path.join(OUTPUT_DIR, 'images')
print('Rendering images at all noise levels...')
render_noisy_images(questions, IMAGE_DIR, noise_levels=NOISE_LEVELS)

In [ ]:
# ── Preview noise levels on a sample problem ──
from PIL import Image

sample_clean = render_text_to_image(questions[0])
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
names = get_noise_level_names()

for idx, level in enumerate(range(10)):
    ax = axes[idx // 5][idx % 5]
    noisy = apply_noise_level(sample_clean, level, text=questions[0], seed=42)
    ax.imshow(noisy)
    ax.set_title(f'L{level}: {names[level]}', fontsize=10)
    ax.axis('off')

plt.suptitle('Noise Levels Applied to Sample Problem', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'noise_levels_preview.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════
#  RUN ABLATION
# ══════════════════════════════════════════════════════════

vlm = VLMModel(**MODEL)
vlm.load()
short = MODEL['name'].split('/')[-1]

# First: get text-only baseline
print('\n── Text-Only Baseline ──')
text_results_path = os.path.join(OUTPUT_DIR, f'{short}_text_baseline.json')
if os.path.exists(text_results_path):
    with open(text_results_path) as f:
        text_baseline = json.load(f)
    print(f'  Loaded from cache: acc={text_baseline["accuracy"]:.3f}')
else:
    text_correct = []
    for q, ref in tqdm(zip(questions, references), total=N, desc='Text'):
        pred = vlm.generate_text_only(q)
        text_correct.append(answers_match(pred, ref))
    text_baseline = {
        'accuracy': compute_accuracy(text_correct),
        'correct': text_correct,
        'ci_95': list(binomial_ci(sum(text_correct), N)),
    }
    with open(text_results_path, 'w') as f:
        json.dump(text_baseline, f)
    print(f'  Accuracy: {text_baseline["accuracy"]:.3f}')

# Run each noise level
all_level_results = {}

for level in NOISE_LEVELS:
    config = NOISE_CONFIGS[level]
    level_name = config['name']
    results_path = os.path.join(OUTPUT_DIR, f'{short}_level_{level}_{level_name}.json')
    
    if os.path.exists(results_path):
        with open(results_path) as f:
            all_level_results[level] = json.load(f)
        print(f'Level {level} ({level_name}): loaded from cache, acc={all_level_results[level]["accuracy"]:.3f}')
        continue
    
    print(f'\n── Level {level}: {level_name} ── {config["description"]}')
    level_dir = os.path.join(IMAGE_DIR, f'level_{level}_{level_name}')
    
    correct_flags = []
    errors = []
    preds = []
    
    for i in tqdm(range(N), desc=f'L{level}'):
        img_path = os.path.join(level_dir, f'q{i:03d}.png')
        try:
            img = Image.open(img_path).convert('RGB')
            pred = vlm.generate_with_image(img)
        except Exception as e:
            pred = f'ERROR: {e}'
        
        preds.append(pred)
        correct_flags.append(answers_match(pred, references[i]))
        errors.append(classify_error(pred, references[i]))
    
    acc = compute_accuracy(correct_flags)
    ci = binomial_ci(sum(correct_flags), N)
    boot_ci = bootstrap_ci(correct_flags)
    
    from collections import Counter
    error_dist = dict(Counter(errors))
    
    level_result = {
        'level': level,
        'name': level_name,
        'description': config['description'],
        'accuracy': acc,
        'correct': correct_flags,
        'errors': errors,
        'error_distribution': error_dist,
        'ci_95': list(ci),
        'boot_ci_95': list(boot_ci),
        'n': N,
    }
    
    with open(results_path, 'w') as f:
        json.dump(level_result, f, indent=2)
    
    all_level_results[level] = level_result
    print(f'  Accuracy: {acc:.3f}  CI: [{ci[0]:.3f}, {ci[1]:.3f}]  Errors: {error_dist}')

vlm.unload()
print('\nAblation complete!')

In [ ]:
# ══════════════════════════════════════════════════════════
#  VISUALIZATION: Accuracy vs Noise Level (Degradation Curve)
# ══════════════════════════════════════════════════════════

levels = sorted(all_level_results.keys())
accs = [all_level_results[l]['accuracy'] * 100 for l in levels]
ci_lows = [all_level_results[l]['ci_95'][0] * 100 for l in levels]
ci_highs = [all_level_results[l]['ci_95'][1] * 100 for l in levels]
names = [all_level_results[l]['name'] for l in levels]
text_acc = text_baseline['accuracy'] * 100

fig, ax = plt.subplots(figsize=(12, 6))

# Error band
ax.fill_between(levels, ci_lows, ci_highs, alpha=0.2, color='#C44E52')
# Main line
ax.plot(levels, accs, 'o-', color='#C44E52', linewidth=2.5, markersize=8,
        label='Image input (noisy)', zorder=5)

# Text-only baseline
text_ci = text_baseline['ci_95']
ax.axhline(y=text_acc, color='#4C72B0', linestyle='--', linewidth=2,
           label=f'Text-only baseline ({text_acc:.1f}%)')
ax.axhspan(text_ci[0]*100, text_ci[1]*100, alpha=0.1, color='#4C72B0')

# Annotate each point
for l, acc, name in zip(levels, accs, names):
    ax.annotate(f'{acc:.1f}%', (l, acc), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(levels)
ax.set_xticklabels([f'L{l}\n{n}' for l, n in zip(levels, names)],
                   fontsize=9, rotation=45, ha='right')
ax.set_xlabel('Noise Level', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title(f'Accuracy Degradation Curve — {short}\n'
             f'How visual noise affects VLM math reasoning (N={N})',
             fontsize=13)
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(text_acc, max(accs)) + 15)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'{short}_degradation_curve.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Error type evolution across noise levels ──
error_cats = ['correct', 'arithmetic_error', 'reasoning_error', 'no_number', 'vision_error']
colors = ['#55A868', '#4C72B0', '#8172B2', '#DDAA33', '#C44E52']

fig, ax = plt.subplots(figsize=(12, 6))

bottoms = np.zeros(len(levels))
for cat, color in zip(error_cats, colors):
    counts = [all_level_results[l]['error_distribution'].get(cat, 0) for l in levels]
    pcts = [c / N * 100 for c in counts]
    ax.bar(levels, pcts, bottom=bottoms, color=color, label=cat.replace('_', ' '),
           edgecolor='white', linewidth=0.5)
    bottoms += np.array(pcts)

ax.set_xticks(levels)
ax.set_xticklabels([f'L{l}\n{all_level_results[l]["name"]}' for l in levels],
                   fontsize=9, rotation=45, ha='right')
ax.set_xlabel('Noise Level', fontsize=12)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title(f'Error Type Evolution Across Noise Levels — {short}', fontsize=13)
ax.legend(fontsize=10, bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'{short}_error_evolution.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary table ──
rows = [{'Level': 'Baseline', 'Name': 'text_only', 'Accuracy': f"{text_acc:.1f}%",
         'CI 95%': f"[{text_baseline['ci_95'][0]*100:.1f}, {text_baseline['ci_95'][1]*100:.1f}]",
         'Drop vs Text': '-'}]
for l in levels:
    r = all_level_results[l]
    drop = r['accuracy'] * 100 - text_acc
    rows.append({
        'Level': l,
        'Name': r['name'],
        'Accuracy': f"{r['accuracy']*100:.1f}%",
        'CI 95%': f"[{r['ci_95'][0]*100:.1f}, {r['ci_95'][1]*100:.1f}]",
        'Drop vs Text': f"{drop:+.1f}pp",
        'Vision Errors': r['error_distribution'].get('vision_error', 0),
        'Arith Errors': r['error_distribution'].get('arithmetic_error', 0),
    })

summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(os.path.join(OUTPUT_DIR, f'{short}_ablation_summary.csv'), index=False)
print(f"\nSaved to {OUTPUT_DIR}/{short}_ablation_summary.csv")